In [2]:
import pandas as pd
from google.oauth2 import service_account
from google.cloud import bigquery

# Set up credentials
credentials = service_account.Credentials.from_service_account_file(
    '../credentials/nhs-dna-analytics-f45003a03f20.json'
)

project_id = 'nhs-dna-analytics'
client = bigquery.Client(credentials=credentials, project=project_id)

print("Connected to BigQuery successfully")
print(f"Project: {project_id}")

Connected to BigQuery successfully
Project: nhs-dna-analytics


In [3]:
# Load all three months
files = {
    '2024-10': '../data/Practice_Level_Crosstab_Oct_24.csv',
    '2024-11': '../data/Practice_Level_Crosstab_Nov_24.csv',
    '2024-12': '../data/Practice_Level_Crosstab_Dec_24.csv'
}

for month, filepath in files.items():
    print(f"Loading {month}...")
    df = pd.read_csv(filepath)
    df['source_month'] = month
    
    table_id = f"{project_id}.raw.appointments_{month.replace('-', '_')}"
    
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        autodetect=True
    )
    
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    
    table = client.get_table(table_id)
    print(f"  Loaded {table.num_rows:,} rows to {table_id}")

print("\nAll months loaded successfully")

Loading 2024-10...
  Loaded 1,070,857 rows to nhs-dna-analytics.raw.appointments_2024_10
Loading 2024-11...
  Loaded 1,030,509 rows to nhs-dna-analytics.raw.appointments_2024_11
Loading 2024-12...
  Loaded 987,680 rows to nhs-dna-analytics.raw.appointments_2024_12

All months loaded successfully
